# 05 — Is the trend evolution or lineage replacement?

A rising overall trend can come from genomes changing **inside** lineages, or from high-value lineages **replacing** low-value ones. Adding lineage fixed effects absorbs the replacement part, leaving the within-lineage slope.

Also covers lineage growth rate, founder composition, and robustness checks.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)
from rsv import io, composition
from rsv import trends

In [2]:
d = pd.read_csv("../data/processed/rsv_features.tsv", sep="\t")
d = d[(d.year>=2015)&(d.year<=2026)]
F = ["gc_content","oe_CG","oe_GC","oe_GG","oe_CC"]
print(len(d), "sequences")

32059 sequences


## Step 1 — decomposition
Opposite signs between `slope_total` and `slope_within` mean the raw trend is an artefact of turnover (a Simpson's paradox).

In [3]:
dec = pd.concat([trends.decompose_trend(s,f).assign(Subtype=sub)
                 for sub,s in d.groupby("Subtype") for f in F], ignore_index=True)
dec[["Subtype","feature","slope_total","slope_within","pct_from_turnover"]].round(6)

,Subtype,feature,slope_total,slope_within,pct_from_turnover
0,A,gc_content,-0.000016,0.000013,179.039876
1,A,oe_CG,-0.000321,-0.000147,54.047262
2,A,oe_GC,-0.000044,0.000259,685.984235
3,A,oe_GG,0.000399,0.000220,44.757321
4,A,oe_CC,-0.000150,-0.000047,68.662140
5,B,gc_content,-0.000005,0.000038,873.984778
6,B,oe_CG,0.000649,-0.000522,180.485672
7,B,oe_GC,0.000400,-0.000076,118.929678
8,B,oe_GG,0.001398,0.000523,62.585687
9,B,oe_CC,-0.000657,0.001170,277.951831


## Per-lineage slopes
Do individual lineages move in the same direction?

In [4]:
sl = trends.per_lineage_slopes(d, "oe_CG")
print(f"{len(sl)} lineages, {sl.sig.sum()} significant")
sl.groupby("Subtype").agg(n=("slope","size"), median_slope=("slope","median"),
                          n_positive=("slope", lambda x:(x>0).sum()))

34 lineages, 22 significant


,n,median_slope,n_positive
Subtype,,,
A,25,-0.000126,12
B,9,-0.000162,3


## Step 3 — lineage growth rate vs composition
Growth rate = logistic change in a lineage's share of its subtype per year.

In [5]:
growth = trends.lineage_growth_rate(d)
tab, corr = trends.growth_vs_composition(d, growth, F)
from statsmodels.stats.multitest import multipletests
corr["p_adj"] = multipletests(corr["p"], method="fdr_bh")[1]
corr.sort_values("p").round(4)

,Subtype,feature,n_lineages,rho,p,p_adj
8,B,oe_GG,14,0.5780,0.0304,0.3026
2,A,oe_GC,32,-0.3343,0.0615,0.3026
6,B,oe_CG,14,0.4418,0.1138,0.3026
0,A,gc_content,32,-0.2797,0.1211,0.3026
9,B,oe_CC,14,-0.2527,0.3833,0.6024
4,A,oe_CC,32,-0.1584,0.3867,0.6024
1,A,oe_CG,32,-0.1463,0.4244,0.6024
5,B,gc_content,14,0.1736,0.5528,0.6024
7,B,oe_GC,14,0.1604,0.5838,0.6024
3,A,oe_GG,32,0.0957,0.6024,0.6024


## Step 4 — founder composition vs later success
Tree-free analogue of the paper's 'earliest viruses' analysis: does a lineage's founding composition predict how successful it later becomes?

In [6]:
fnd = trends.founder_composition(d, F)
_, fres = trends.founder_vs_success(fnd, growth, F)
fres.head(10).round(4)

,Subtype,feature,outcome,n_lineages,rho,p,p_adj
18,B,oe_CG,growth_rate,14,0.6879,0.0065,0.1962
24,B,oe_GG,growth_rate,14,0.5868,0.0274,0.4108
0,A,gc_content,growth_rate,32,-0.3409,0.0562,0.5621
9,A,oe_GG,growth_rate,32,0.2493,0.1689,0.7667
6,A,oe_GC,growth_rate,32,-0.2207,0.2249,0.7667
3,A,oe_CG,growth_rate,32,-0.2192,0.2280,0.7667
17,B,gc_content,span_years,14,0.3233,0.2595,0.7667
27,B,oe_CC,growth_rate,14,-0.3231,0.2599,0.7667
21,B,oe_GC,growth_rate,14,0.3099,0.2809,0.7667
4,A,oe_CG,peak_share,32,-0.1848,0.3114,0.7667


## Step 5 — robustness

In [7]:
rob = trends.robustness(d, "oe_CG")
rob[["Subtype","scenario","n","slope_total","p_total","slope_within","p_within"]].round(6)

,Subtype,scenario,n,slope_total,p_total,slope_within,p_within
0,A,full,15052,-0.000321,0.0,-0.000147,0.000001
1,A,excl_2020_2021,13148,-0.000315,0.0,0.000001,0.973938
2,A,region_adjusted,15052,-0.000301,0.0,-0.000129,0.000025
3,A,region_balanced,9161,-0.000230,0.0,0.000053,0.158588
4,B,full,17007,0.000649,0.0,-0.000522,0.000000
5,B,excl_2020_2021,15378,0.000602,0.0,-0.000540,0.000000
6,B,region_adjusted,17007,0.000494,0.0,-0.000566,0.000000
7,B,region_balanced,8051,0.000740,0.0,-0.000324,0.000000
